# 40 — приёмка трёх CLI, которых не хватало контуру D3

Ноутбук проверяет ровно то, что добавил PR: `scripts/train_ft_judge.py`,
`run_m3.py --prompt-style perchunk`, `score.py --workers/--m3-concurrency`.

| Что | Где считается | Нужен GPU | Нужен vLLM |
|---|---|:-:|:-:|
| Юнит-тесты новых модулей | `tests/` | нет | нет |
| Симметрия формата и баланс классов | `train_ft_judge.py --smoke-only` | нет | нет |
| Пофрагментная верификация | `run_m3.py`, `score.py` | да | да |
| Параллельный скоринг | `score.py --workers` | да | да |
| Обучение судьи на фолде | `train_ft_judge.py` | да | нет |

Ноутбук — пусковая установка: ни одной строки расчёта в ячейках. Логика живёт
в репозитории, иначе она не попадает ни в `run.yaml`, ни в git-хэш.

**Что прислать обратно:** вывод последней ячейки («сводка приёмки») целиком.

## 0. Конфигурация

In [ ]:
import json
import os
import subprocess
import sys
import time

REPO = "/home/jupyter/datasphere/project/rag-reliability"
GIT_URL = "https://github.com/MurkaSelebry/rag-reliability.git"
BRANCH = "integration"  # база; после мержа PR проверка пойдёт прямо с неё
VERIFY_BRANCH = "task/D5-cli-gaps"  # ветка проверяемого PR

MODEL = "Qwen/Qwen2.5-7B-Instruct"
API_BASE = "http://localhost:8000/v1"
DATA = "data/alfa.jsonl"
FOLDS = "data/splits/folds_alfa.json"
OUT = "predictions/verify"

# vLLM не проверяет ключ, но клиент openai требует непустую строку.
os.environ["OPENAI_API_KEY"] = "EMPTY"

VERDICT = {}
print(sys.version)
print(f"repo={REPO}\nbranch={VERIFY_BRANCH}\nmodel={MODEL}")

## 1. Репозиторий и окружение

In [ ]:
!test -d {REPO} || git clone --branch {BRANCH} {GIT_URL} {REPO}
!cd {REPO} && git fetch --quiet origin {VERIFY_BRANCH} && git checkout --quiet {VERIFY_BRANCH} && git pull --quiet origin {VERIFY_BRANCH}
!cd {REPO} && git log --oneline -1 && git status --short | head

In [ ]:
!cd {REPO} && pip install -q -e ".[dev,cloud,ft]"

In [ ]:
import torch

CUDA = torch.cuda.is_available()
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1) if CUDA else 0.0
print(f"cuda={CUDA} device={torch.cuda.get_device_name(0) if CUDA else 'cpu'} vram={VRAM_GB} GB")
print(f"torch={torch.__version__}")
VERDICT["vram_gb"] = VRAM_GB

## 2. Без GPU: тесты и симметрия формата

Первое, что ловится здесь, — расхождение обучающего формата с форматом
инференса. Судья, обученный на одном шаблоне и опрошенный другим, теряет якорь
вердикта: вероятность приходит уже не из логпробов, а из regex-ветки, и прогон
выглядит успешным. Проверка идёт по всем 4466 примерам корпуса, до загрузки
весов.

In [ ]:
!cd {REPO} && python -m pytest tests/test_ft_judge.py tests/test_m3_perchunk.py tests/test_m3_perchunk_cli.py tests/test_score_cli.py -q

In [ ]:
!cd {REPO} && python scripts/train_ft_judge.py --smoke-only --fold 0 --data {DATA} --folds {FOLDS} 2>&1 | head -12

## 3. vLLM

Поднимается один раз на весь ноутбук: разделы 4 и 5 ходят в него, раздел 6 —
нет (обучение сервер не требует).

In [ ]:
import requests

VLLM_LOG = "/tmp/vllm.log"
vllm = subprocess.Popen(
    [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--port", "8000",
        "--max-model-len", "16384",
        "--gpu-memory-utilization", "0.85",
    ],
    cwd=REPO,
    stdout=open(VLLM_LOG, "w"),
    stderr=subprocess.STDOUT,
)

VERDICT["vllm_ready"] = False
for attempt in range(180):
    try:
        if requests.get(f"{API_BASE}/models", timeout=2).status_code == 200:
            VERDICT["vllm_ready"] = True
            print(f"vLLM готов за ~{attempt * 5} с")
            break
    except requests.RequestException:
        pass
    time.sleep(5)

if not VERDICT["vllm_ready"]:
    print(f"vLLM не поднялся, смотри {VLLM_LOG}")
    !tail -30 {VLLM_LOG}

### Смоук на logprobs

Без него отладка судьи бессмысленна: основная ветка кода живёт на логпробах, а
их отсутствие вырождается в 0.5 для всех кейсов — молча.

In [ ]:
!cd {REPO} && python scripts/run_m3.py --data {DATA} --output {OUT}/logprob_smoke.jsonl --prompt-style axes --backend openai_judge --model {MODEL} --api-base {API_BASE} --cache-dir results/verify/cache --limit 5

In [ ]:
rows = [json.loads(line) for line in open(f"{REPO}/{OUT}/logprob_smoke.jsonl", encoding="utf-8")]
methods = sorted({row["prob_method"] for row in rows})
faiths = [round(row["scores"]["m3.p_faith"], 3) for row in rows]
print(f"prob_method={methods}")
print(f"p_faith={faiths}")
VERDICT["logprobs"] = methods == ["logprobs"]
VERDICT["p_faith_spread"] = max(faiths) - min(faiths)

## 4. Пофрагментная верификация faithfulness

Каждый запрос видит ровно один чанк. Ось relevance чанки не получает по
контракту C3 — её определение опирается только на вопрос и ответ, — поэтому
вероятностей осей в артефакте нет вовсе, есть пять фич.

In [ ]:
!cd {REPO} && python scripts/run_m3.py --data {DATA} --output {OUT}/perchunk_smoke.jsonl --prompt-style perchunk --backend openai_judge --model {MODEL} --api-base {API_BASE} --cache-dir results/verify/cache --concurrency 8 --limit 10

In [ ]:
!cd {REPO} && python scripts/score.py --method m3_perchunk --variant verify --data {DATA} --output {OUT}/m3_perchunk/scores.jsonl --model {MODEL} --m3-backend openai_judge --m3-api-base {API_BASE} --m3-cache-dir results/verify/cache --m3-concurrency 8 --workers 4 --limit 30

In [ ]:
rows = [json.loads(line) for line in open(f"{REPO}/{OUT}/m3_perchunk/scores.jsonl", encoding="utf-8")]
keys = sorted(rows[0]["scores"])
print(f"кейсов: {len(rows)}")
print(f"ключи: {keys}")
for row in rows[:5]:
    print(row["id"], {key: round(value, 3) for key, value in sorted(row["scores"].items())})

VERDICT["perchunk_n"] = len(rows)
VERDICT["perchunk_keys"] = keys
# Пофрагментный сигнал обязан различать кейсы: константа означала бы, что
# судья не смотрит на чанк, а отвечает на промпт.
maxima = [row["scores"]["m3.max_chunk_score"] for row in rows]
VERDICT["perchunk_spread"] = round(max(maxima) - min(maxima), 4)
print(f"разброс m3.max_chunk_score: {VERDICT['perchunk_spread']}")

## 5. Параллельный скоринг

`--workers` меряется на чистом кэше в обе стороны: на общем кэше второй прогон
был бы мгновенным независимо от параллельности.

In [ ]:
TIMES = {}
for workers in (1, 16):
    started = time.time()
    subprocess.run(
        [
            sys.executable, "scripts/score.py",
            "--method", "m3_openai_judge",
            "--variant", f"verify_w{workers}",
            "--data", DATA,
            "--output", f"{OUT}/workers_{workers}/scores.jsonl",
            "--model", MODEL,
            "--m3-backend", "openai_judge",
            "--m3-api-base", API_BASE,
            "--m3-cache-dir", f"results/verify/cache_w{workers}",
            "--workers", str(workers),
            "--limit", "40",
        ],
        cwd=REPO,
        check=True,
    )
    TIMES[workers] = round(time.time() - started, 1)
    print(f"workers={workers}: {TIMES[workers]} c")

VERDICT["workers_times"] = TIMES
VERDICT["workers_speedup"] = round(TIMES[1] / max(TIMES[16], 1e-9), 2)
print(f"ускорение: x{VERDICT['workers_speedup']}")

In [ ]:
# Порядок обязателен: --resume дочитывает файл сверху, и строки «как
# посчиталось» оставили бы дыру, которую перезапуск не заметит.
ids_1 = [json.loads(line)["id"] for line in open(f"{REPO}/{OUT}/workers_1/scores.jsonl", encoding="utf-8")]
ids_16 = [json.loads(line)["id"] for line in open(f"{REPO}/{OUT}/workers_16/scores.jsonl", encoding="utf-8")]
VERDICT["workers_same_order"] = ids_1 == ids_16
print(f"порядок совпал: {VERDICT['workers_same_order']} ({len(ids_1)} кейсов)")

## 6. Обучение судьи на фолде

Смоук: фолд 0, 60 кейсов, одна эпоха, LoRA r=16, контекст 1024 — минут на 10.
Ассерт VRAM снят флагом `--allow-small-gpu` только потому, что это смоук;
отчётный прогон идёт через `jobs/ft_judge_fold*.yaml` без него.

Сервер vLLM обучению не нужен и занимает память — гасим.

In [ ]:
vllm.terminate()
vllm.wait(timeout=120)
torch.cuda.empty_cache()
print("vLLM остановлен")

In [ ]:
!cd {REPO} && python scripts/train_ft_judge.py --data {DATA} --folds {FOLDS} --fold 0 --model {MODEL} --mode direct --tuning lora --lora-r 16 --lora-alpha 32 --max-length 1024 --epochs 1 --batch-size 1 --grad-accum 4 --pos-weight-mode balanced --oversample-negatives --save-strategy epoch --save-total-limit 1 --seed 42 --limit 60 --variant verify_smoke --output-dir results/verify/ft_judge --predictions-output {OUT}/ft_judge/scores.jsonl --diagnostics-output {OUT}/ft_judge/ft_diagnostics.json

In [ ]:
diagnostics = json.load(open(f"{REPO}/{OUT}/ft_judge/ft_diagnostics.json", encoding="utf-8"))
rows = [json.loads(line) for line in open(f"{REPO}/{OUT}/ft_judge/scores.jsonl", encoding="utf-8")]

print(f"кейсов в артефакте: {len(rows)}; ключи: {sorted(rows[0]['scores'])}")
print(f"pos_weight={diagnostics['pos_weight']:.4f} class_balance={diagnostics['class_balance']}")
print(f"collapsed={diagnostics['collapsed']} reason={diagnostics['collapse_reason']}")
for log in diagnostics["epochs"]:
    print(
        f"  эпоха {log['epoch']}: const_share={log['const_share']:.4f} "
        f"entropy={log['output_entropy']:.4f} degenerate={log['is_degenerate']}"
    )

VERDICT["ft_n"] = len(rows)
VERDICT["ft_keys"] = sorted(rows[0]["scores"])
VERDICT["ft_collapsed"] = diagnostics["collapsed"]
VERDICT["ft_const_share"] = round(diagnostics["const_share"], 4)
VERDICT["ft_epochs_logged"] = len(diagnostics["epochs"])

In [ ]:
# Возобновляемость: второй запуск с --resume не должен переобучать эпоху заново.
started = time.time()
subprocess.run(
    [
        sys.executable, "scripts/train_ft_judge.py",
        "--data", DATA, "--folds", FOLDS, "--fold", "0",
        "--model", MODEL, "--mode", "direct", "--tuning", "lora",
        "--lora-r", "16", "--lora-alpha", "32", "--max-length", "1024",
        "--epochs", "1", "--batch-size", "1", "--grad-accum", "4",
        "--limit", "60", "--variant", "verify_smoke", "--resume",
        "--output-dir", "results/verify/ft_judge",
        "--predictions-output", f"{OUT}/ft_judge_resume/scores.jsonl",
        "--diagnostics-output", f"{OUT}/ft_judge_resume/ft_diagnostics.json",
    ],
    cwd=REPO,
    check=True,
)
VERDICT["ft_resume_sec"] = round(time.time() - started, 1)
print(f"--resume отработал за {VERDICT['ft_resume_sec']} c")

## 7. Сводка приёмки

In [ ]:
CHECKS = [
    ("VRAM >= 70 GB", VERDICT.get("vram_gb", 0) >= 70),
    ("vLLM поднялся", VERDICT.get("vllm_ready") is True),
    ("вероятности из logprobs", VERDICT.get("logprobs") is True),
    ("вероятности различаются", VERDICT.get("p_faith_spread", 0) > 0.01),
    ("perchunk: 5 фич", len(VERDICT.get("perchunk_keys", [])) == 5),
    ("perchunk: сигнал не константа", VERDICT.get("perchunk_spread", 0) > 0.01),
    ("workers: тот же порядок", VERDICT.get("workers_same_order") is True),
    ("workers: есть ускорение", VERDICT.get("workers_speedup", 0) > 2),
    ("FT: артефакт непустой", VERDICT.get("ft_n", 0) > 0),
    ("FT: ключи m3.p_faith/p_rel", VERDICT.get("ft_keys") == ["m3.p_faith", "m3.p_rel"]),
    ("FT: диагностика по эпохам", VERDICT.get("ft_epochs_logged", 0) >= 1),
    ("FT: не схлопнулся", VERDICT.get("ft_collapsed") is False),
]

print("=" * 60)
for label, ok in CHECKS:
    print(f"{'OK  ' if ok else 'FAIL'}  {label}")
print("=" * 60)
print(json.dumps(VERDICT, ensure_ascii=False, indent=2))
print("=" * 60)
print(f"Итого: {sum(1 for _, ok in CHECKS if ok)}/{len(CHECKS)}")